# CLIP-H Length-of-Stay Runner — full pipeline

**Goal:** generate and curate a small set of human-readable clinical **hypotheses** that predict a
**prolonged hospital length of stay** (hospital LOS > 72 hours - strictly more than 3 days, measured in hours; cohort restricted to stays >= 30h),
using only information available in the **first 24 hours** of admission, run a strict clinical
**critique review**, then **annotate held-out profiles and evaluate** the final hypotheses. Only the conversational review (Phase 2C export, Phase 2C+ import) is manual.

This runner reuses the same CLIP-H machinery (local embedding + SAE + LLM interpretation +
clinical critique) as the other tasks; only the **task framing and leakage rules** change: the
outcome is the length of stay, so every feature must be knowable by hour 24 and must never
reference the stay length, discharge, or death.

### Method in one paragraph
Each admission is rendered as a short first-24h clinical paragraph and embedded by a **local,
open-weight** model (Qwen3-Embedding-0.6B). A **Top-K sparse autoencoder (SAE)** learns reusable
concept neurons from those embeddings; an LLM names the most predictive neurons as one-sentence
hypotheses; the candidates are **agentically self-refined against data-driven feedback**
(Self-Refine; Madaan et al., 2023) on a discovery split, a clinical **critic** reviews them, redundant
concepts are merged by **firing-vector de-duplication**, and a **generalizability gate** (prevalence
floor + BH-FDR + bootstrap stability on the held-out confirmation split) drops unstable ones; finally a simple model evaluates the curated hypotheses on held-out data so each
prediction decomposes into human-readable reasons.

### Map of this notebook
Run the cells top to bottom. Every step first looks for its saved file; if the file is there the step loads it and makes no call. `FORCE_RERUN` (Settings) redoes a step.

| # | Cell | What it does | Reads | If its file exists | If not | Writes | LLM calls |
|---|------|--------------|-------|--------------------|--------|--------|-----------|
| 1 | Settings | Every setting of the run | | | | | none |
| 2 | Setup | Finds the run folder, makes result folders, API key, call cache, usage log | `code/`, `prompts/` | | | `caches/usage_log.jsonl`, `caches/llm_call_cache.jsonl` (as calls happen) | none |
| 3 | Phase 0 | Loads profiles and labels, subsamples, splits TRAIN into discovery and confirmation | `train.jsonl`, `test.jsonl` | loads them | builds them from `*_profile.jsonl` + `*_labels.csv` | `run_manifest.json` | none |
| 4 | Phase 1 | Embeddings, SAE, neuron selection, strong / weak split | `phase1_sae/embeddings.npz`, `sae.pkl` (of this run, or of `PHASE1_FROM`) | loads both | embeds with the local model; trains the SAE | the two files | none (local model) |
| 5 | Phase 2A | 3 candidate interpretations per strong neuron, fidelity score, best one kept | `phase2a_interpret/hypotheses_pool.csv` | loads it | interprets and scores | that file | `INTERP_MODEL`, `ANNOT_MODEL` |
| 6 | Phase 2A+ | Names the weak neurons, groups related ones into OR-hypotheses | `pooled_group.json` | loads it | names and groups | `pooled_group.json`, adds rows to the pool | `GROUP_MODEL` |
| 7 | Fidelity gate | Drops hypotheses below `MIN_FIDELITY`, drops exact duplicates | `hypotheses_curated_candidates.csv`, `curation_summary.json` | loads them | computes | those two files | none |
| 8 | Phase 2B | Self-refine on the discovery sample: critic and refiner, up to `REFINE_ITERS` rounds | `phase2b_self_refine/self_refine_hypotheses.csv` | loads it | runs the loop | that file, `self_refine_trace.json` | `ANNOT_MODEL`, `REFINE_MODEL` |
| 9 | Phase 2C | Writes the review prompt for a chat model (**manual step follows**) | `train_features.csv`, `prompts/review.txt` | always rewritten | | `phase2c_review/CONVO_01_critique_prompt.md` | `ANNOT_MODEL` only for labels not yet cached |
| 10 | Phase 2C+ | Checks and applies the pasted review reply | `CONVO_02_critique_response.json` | reads it | **stops and tells you to do the manual step** | `hypotheses.csv`, `hypotheses_conversational_merged.csv`, `conversational_curation_summary.json` | none |
| 11 | Phase 2D | Merges hypotheses that are true for the same patients | (labels from the cache) | always recomputed | | `phase2d_dedup/dedup_pairs.csv`, `dedup_clusters.csv` | `ANNOT_MODEL`, `DEDUP_MODEL` for borderline pairs |
| 12 | Phase 2E | Confirmation gate: FDR + bootstrap stability on the confirmation split | (labels from the cache) | always recomputed | | `phase2e_confirm/confirmation_gate_tests.csv`, `confirmation_gate_summary.json` | `ANNOT_MODEL` (Batch API on the GPT arm) |
| 13 | Phase 3A | Annotates TEST and TRAIN, fits the prediction model, association tests | `phase3_prediction/annotation_matrix.npz`, `hypothesis_evaluation.csv`, `annotation_metrics.json` | loads them | annotates the test set | those, plus `heldout_metrics.json`, `hypothesis_association_*.{csv,json}`, `model_metrics.csv` | `ANNOT_MODEL` (Batch API on the GPT arm) |
| 14 | Phase 3B | Checks that every expected file exists, prints the summary and the cost | all of the above | | stops and lists what is missing | `task_summary.json`, `results_index.json` | none |

**Calls are never paid twice.** Every reply is saved in `caches/llm_call_cache.jsonl` and every label in a `caches/*_annotation_cache.json` file as soon as it arrives, so an interrupted cell can simply be run again.

**What you see while a cell runs.** A step with many calls prints one line every 15 seconds:
`[Phase 2E annotation] 320 of 1500 (21%) | 1:12 elapsed, about 4:30 left | requests 960, from cache 0, retries 2, failed 0`.
Problems are printed as they happen: lines starting with `[llm]` (a retry, a cut-off reply), `[annotation]` (a reply that could not be read and is asked for again), `NOTE:` (labels asked one at a time) and `WARNING:` (something could not be obtained; for labels the count is also saved in `caches/annotation_failures.json`). A bad key, an empty balance or an unknown model stops the cell with `FatalAPIError`.

**What you see when a cell ends.** One line with the count in and out, then one table with a row per hypothesis. The columns are the same in every phase: `#`, `hypothesis`, `from`, the numbers of that phase, `status`, `reason`. `from` is where the hypothesis came from: `N339` = neuron 339, `G2` = weak-neuron group 2, `R1` = added by the reviewers, `N12+N88` = merged from two. Dropped rows are grey and at the bottom. Phase 3B repeats the counts of all phases and the held-out result (value, 95% interval, what to compare it with).

**The two runner notebooks** (`los_runner_gpt.ipynb`, `los_runner_deepseek.ipynb`) differ in one line: `MODEL_PROFILE` in Settings.

### Where things are
- Input data: `data/hosp_los3/` at the repository root (`TASK_DIR`, setting `TASK_DATA_REL`): `*_profile.jsonl`, `*_labels.csv`, `*_features.csv` per split
- Code: `code/` · Prompts: `prompts/` (one file per step; `prompts/README.md` lists the sections)
- Results: `results/<RUN_NAME>/phase1_sae/` (shared by both arms) and `results/<RUN_NAME>/los_<arm>/` with one folder per phase and `caches/`. A new `RUN_NAME` starts a new, empty run folder; it reads the saved replies and labels of the other runs (`REUSE_CACHES_FROM`)


## 1. Settings
The one place to edit. Each line says what it controls and which phase reads it.

In [ ]:
import os
# Set before torch is imported: on Apple Silicon, operations that MPS lacks fall back to the CPU.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# ---- Run ---------------------------------------------------------------------------------
RUN_FOLDER = None           # this task folder (the one holding code/ and prompts/). None = found from the working
                            # directory; set a path (relative or absolute) when the notebook is started elsewhere
RUN_NAME = 'clip_full'       # every result of this run goes to results/<RUN_NAME>/. A new name = a new, empty run folder
PHASE1_FROM = None          # name of a run whose embeddings and SAE this run uses (read, not rewritten unless FORCE_RERUN says so);
                            # None = this run makes its own. Same embeddings + SAE = same neurons, which an ablation usually wants.
REUSE_CACHES_FROM = 'all'   # other runs whose saved replies and labels this run may read (never written to), same arm only.
                            # 'all' = every other run folder in results/; a list of run names = only those; [] = none.
                            # A reply is reused only for the same model and the same prompt, so results are unaffected.
                            # Example: REUSE_CACHES_FROM = ['clip_full', 'my_first_ablation']
TASK_NAME = 'los'           # the arm's results go to results/<RUN_NAME>/<TASK_NAME>_<MODEL_PROFILE>/
TASK_DATA_REL = '../../data/hosp_los3'   # <repo>/data/hosp_los3/: train/test _profile.jsonl, _labels.csv, _features.csv, relative to the run folder
TASK_DESCRIPTION, POS_LABEL = 'prolonged hospital length of stay', 'LOS>3 days'   # told to the 2B critic and the 2D confirmer
POS_NAME, NEG_NAME = 'long stay', 'not long stay'   # the two outcome classes, as named in printed lines and table columns
RAW_FACTS = 'los_raw_facts'   # how a patient's raw values are printed for the reviewer (Phase 2C): a function of
                              # conversational_review.py: 'los_raw_facts' | 'icu_raw_facts' | 'readmission_raw_facts'
TASK_INSTRUCTION = ''       # extra text for the {task_specific_instructions} slot of the interpretation prompt (the prompt is complete without it)
SEED = 42                   # SAE training, subsampling, the discovery/confirmation split, the 2B and 2D calls

# ---- Models: the ONLY line that differs between the two runner notebooks is MODEL_PROFILE ----
MODEL_PROFILE = 'deepseek'      # 'gpt' | 'deepseek'
_MODEL_PROFILES = {
    'gpt':      dict(gen='gpt-4.1', annot='gpt-4.1-mini', refine='gpt-5', dedup='gpt-5', base_url=None,
                     annotate_style='checklist', annotate_workers=10, interp_max_tokens=100, interp_workers=2,
                     group_max_tokens=2000, timeout=300.0, batch_api=True),
    'deepseek': dict(gen='deepseek-v4-pro', annot='deepseek-v4-flash', refine='deepseek-v4-pro', dedup='deepseek-v4-flash',
                     base_url='https://api.deepseek.com',
                     annotate_style='labels', annotate_workers=20,
                     interp_max_tokens=8000,   # limits, not a spend. Every DeepSeek call runs with thinking off (llm_client.py); a model name
                     group_max_tokens=8000,    # ending in '-think' (gen='deepseek-v4-pro-think') keeps the thinking pass for that role
                     interp_workers=16,        # parallel interpretation calls
                     timeout=600.0,            # not used by DeepSeek models: their calls are streamed, the limit is 90 s between two chunks
                     batch_api=False),
}
_PROFILE = _MODEL_PROFILES[MODEL_PROFILE]
INTERP_MODEL = _PROFILE['gen']       # Phase 2A interpretation
GROUP_MODEL = _PROFILE['gen']        # Phase 2A+ weak-neuron names and groups
ANNOT_MODEL = _PROFILE['annot']      # every annotation: 2A fidelity, 2B, 2C exemplars, 2D, 2E, 3A
REFINE_MODEL = _PROFILE['refine']    # Phase 2B refiner and critic (the 2C review is a manual chat, no model here)
DEDUP_MODEL = _PROFILE['dedup']      # Phase 2D confirmer of borderline pairs
EMB_MODEL = 'Qwen/Qwen3-Embedding-0.6B'   # local embedding model, no API
LLM_TIMEOUT = _PROFILE['timeout']    # seconds for one whole request, for every LLM call that does not set its own
ANNOTATE_STYLE = _PROFILE['annotate_style']       # reply format of the annotation call: 'checklist' | 'labels' (prompts/annotation.txt)
ANNOTATE_WORKERS = _PROFILE['annotate_workers']   # parallel annotation calls
ANNOTATION_BATCH_API = _PROFILE['batch_api']      # True: the annotation of Phases 2B to 2E and 3A goes through the OpenAI Batch API
                                                  # (half price; each waits for its batch, up to 24 h). OpenAI only.

# ---- Data (Phase 0) ------------------------------------------------------------------------
N_PATIENTS_TRAIN = 5000   # label-stratified subsample, seeded with SEED (None = all)
N_PATIENTS_TEST = 1000

# ---- SAE and neuron selection (Phase 1) ----------------------------------------------------
SAE_M, SAE_K = 1024, 16        # neurons, and how many are active per profile
SAE_EPOCHS = 1000
SAE_PATIENCE = 50
N_HYPOTHESIS_POOL = 96         # neurons selected; the only count setting (there is no quota on the final set)
SELECTION_METHOD = 'diverse'   # 'diverse' (outcome-guided, de-correlated) | 'lasso' | 'separation_score' | 'correlation'
MMR_LAMBDA = 0.7               # 'diverse' only: higher = more diverse
SPARSE_MAX_PROFILES = 15       # a selected neuron firing on fewer training profiles than this is "weak" (grouped in Phase 2A+)

# ---- Phase 2A ------------------------------------------------------------------------------
N_CANDIDATES = 3               # candidate interpretations written per strong neuron; the one with the best fidelity F1 is kept
MIN_FIDELITY = 0.30            # drop a hypothesis whose fidelity F1 against its neuron is below this

# ---- Phase 2B: self-refine (Madaan et al. 2023, arXiv:2303.17651) --------------------------
CONTROL_LOOP = True            # False skips Phase 2B
REFINE_ITERS = 3               # maximum rounds
SR_MIN_PREV = 0.02             # a hypothesis true for less than 2% of the sample is flagged as too rare
DISCOVERY_FRAC = 0.7           # share of TRAIN for discovery (2B, 2C, 2D); the rest is the confirmation split (2E)
STRATIFY_SPLIT = True          # True: each outcome class is split 70/30, so both parts keep the outcome rate. False: one random
                               # split of all rows. Changing it changes who is in each part, so saved labels no longer cover the split.
SR_SAMPLE_N = 3500             # discovery profiles annotated in 2B, 2C and 2D (None = all); 3,500 is the whole discovery part of 5,000

# ---- Phase 2C: review ----------------------------------------------------------------------
N_REVIEWERS = 3
MIN_CLINICAL_VALIDITY = 1      # keep a reviewed hypothesis with clinical validity at or above this (0-3)
LEAKAGE_DROP = 2               # drop a reviewed hypothesis with leakage risk at or above this (0-2)
WITHHOLD_RAW_FIELDS = set()    # feature columns hidden from the reviewer's raw-value exemplars, e.g. {'hemoglobin'}

# ---- Phase 2D: dedup -----------------------------------------------------------------------
DEDUP_CORR_THR = 0.90          # phi at or above this: the pair is considered for a merge (the LLM confirms or refuses)
DEDUP_AUTO_THR = 0.95          # phi at or above this: merged without asking

# ---- Phase 2E: confirmation gate -----------------------------------------------------------
RUN_TRAIN_GATE = True          # False forwards every hypothesis
GATE_SAMPLE_N = None           # confirmation profiles annotated for the gate (None = all)
GATE_FDR_ALPHA = 0.10          # Benjamini-Hochberg FDR level
GATE_BOOTSTRAP = 1000          # bootstrap resamples (no LLM calls)
GATE_STABILITY = 0.60          # share of resamples in which the hypothesis must stay significant with the same sign
ESTIMABILITY_FLOOR = 15        # the hypothesis must be true for at least this many confirmation profiles
MIN_SURVIVORS = 5              # if fewer pass, the largest-effect ones are added up to this number
GATE_TOP_K = None              # None = the certified set only; a number fills the set up to it by lowest q-value

# ---- Phase 3A ------------------------------------------------------------------------------
GATE_TRAIN_N = 5000            # training profiles annotated to fit the prediction model (None = all)

# ---- What to redo: False reuses the saved file of that step --------------------------------
FORCE_RERUN = {
    'handoff_jsonl': False,   # Phase 0: train.jsonl / test.jsonl
    'embeddings': False,      # Phase 1: embeddings.npz
    'sae': False,             # Phase 1: sae.pkl
    'hypotheses': False,      # Phase 2A, 2A+ and the fidelity gate (LLM calls)
    'self_refine': False,     # Phase 2B (LLM calls)
    'annotation': False,      # Phase 3A test-set tables; labels already in the caches are reused either way
}

print('Settings loaded. Profile:', MODEL_PROFILE, '| interpret / annotate / refine / dedup:', INTERP_MODEL, '/', ANNOT_MODEL, '/',
      REFINE_MODEL, '/', DEDUP_MODEL, '| SAE M/K:', SAE_M, SAE_K, '| pool:', N_HYPOTHESIS_POOL)

## 2. Setup
No computation and no LLM call. Start the notebook from inside this task folder, or set `RUN_FOLDER` in Settings. The API key is asked for once and kept only in this session.

In [ ]:
import sys, json, pickle, getpass
from collections import Counter
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd

# ---- 1. Find the run folder: the one holding code/code_paths.py ----------------------------
def is_run_root(path):
    return (Path(path) / 'code' / 'code_paths.py').exists()


_candidates = [Path(RUN_FOLDER).expanduser()] if RUN_FOLDER else [Path.cwd(), *Path.cwd().resolve().parents]
RUN_FOLDER = next((path.resolve() for path in _candidates if is_run_root(path)), None)
if RUN_FOLDER is None:
    raise FileNotFoundError('Run folder not found. Start the notebook from inside this task folder (the one holding '
                            'code/ and prompts/), or set RUN_FOLDER in Settings to its path.')
TASK_DIR = (RUN_FOLDER / TASK_DATA_REL).resolve()
_missing = [f for f in ('train_profile.jsonl', 'test_profile.jsonl') if not (TASK_DIR / f).exists()]
if _missing:
    raise FileNotFoundError(f'{_missing} not found in {TASK_DIR.resolve()}. Run mimic_cleaning/ first (it writes '
                            'data/<task>/ at the repository root), or set TASK_DATA_REL to your own folder.')

# ---- 2. Result folders ---------------------------------------------------------------------
RUN_DIR = RUN_FOLDER / 'results' / RUN_NAME            # one folder per run
PHASE1_DIR = RUN_FOLDER / 'results' / (PHASE1_FROM or RUN_NAME) / 'phase1_sae'   # embeddings + SAE, shared by both arms
RESULTS_DIR = RUN_DIR / f'{TASK_NAME}_{MODEL_PROFILE}'      # everything else, one folder per arm
P2A_DIR = RESULTS_DIR / 'phase2a_interpret'
P2B_DIR = RESULTS_DIR / 'phase2b_self_refine'
P2C_DIR = RESULTS_DIR / 'phase2c_review'
P2D_DIR = RESULTS_DIR / 'phase2d_dedup'
P2E_DIR = RESULTS_DIR / 'phase2e_confirm'
P3_DIR = RESULTS_DIR / 'phase3_prediction'
CACHE_DIR = RESULTS_DIR / 'caches'                                           # every cache and the usage log
for _dir in (PHASE1_DIR, P2A_DIR, P2B_DIR, P2C_DIR, P2D_DIR, P2E_DIR, P3_DIR, CACHE_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
print('Run folder:', RUN_FOLDER, '\nTask data: ', TASK_DIR, '\nResults:   ', RESULTS_DIR)

# ---- 3. Code: code/ holds one folder per step; code_paths makes them importable --------
CODE_DIR = RUN_FOLDER / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import code_paths

# ---- 4. API key: asked for once, kept only in this session's environment --------------------
_base_url = _PROFILE['base_url']
if _base_url:   # DeepSeek: an OpenAI-compatible endpoint
    os.environ['DEEPSEEK_API_KEY'] = os.environ.get('DEEPSEEK_API_KEY') or getpass.getpass('DeepSeek API key: ')
    os.environ['OPENAI_KEY_SAE'] = os.environ['DEEPSEEK_API_KEY']
    os.environ['OPENAI_BASE_URL_SAE'] = _base_url
else:
    if 'OPENAI_KEY_SAE' not in os.environ:
        os.environ['OPENAI_KEY_SAE'] = os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
    os.environ['OPENAI_BASE_URL_SAE'] = 'https://api.openai.com/v1'
os.environ['EMB_CACHE_DIR'] = str(RUN_FOLDER / 'results' / f'{TASK_NAME}_emb_cache')   # local embedding chunks, shared by all runs
print('Provider:', os.environ['OPENAI_BASE_URL_SAE'])

# ---- 5. LLM client, call cache, usage log (code/common/llm_client.py) ------------------
import llm_client
from llm_client import set_default_timeout, install_usage_meter, install_call_cache, usage_report
from prompts import load_prompt   # reads prompts/<file>.txt
from clip_h_data import read_json, write_json, json_safe
import phase_tables as PT        # the table at the end of each phase (code/common/phase_tables.py)
ORIGIN, FUNNEL = {}, {}          # where each hypothesis came from (N339 = neuron 339, G2 = weak group 2, R1 = reviewer);
                                 # one line per phase for the summary. Both are filled by the phase tables.

set_default_timeout(LLM_TIMEOUT)
install_usage_meter(CACHE_DIR / 'usage_log.jsonl')        # one row per request; cost table: usage_report(CACHE_DIR / 'usage_log.jsonl')
# Every reply is kept; a repeated call is answered from this file. Caches of the runs named in
# REUSE_CACHES_FROM (same arm) are read as well and never written to.
if REUSE_CACHES_FROM == 'all':   # every other run that has caches for this arm
    _reuse_names = sorted(p.name for p in (RUN_FOLDER / 'results').iterdir()
                          if p.name != RUN_NAME and (p / f'{TASK_NAME}_{MODEL_PROFILE}' / 'caches').is_dir())
else:
    _reuse_names = list(REUSE_CACHES_FROM)
_reuse_dirs = [RUN_FOLDER / 'results' / name / f'{TASK_NAME}_{MODEL_PROFILE}' / 'caches' for name in _reuse_names]
for _d in _reuse_dirs:
    if not _d.exists():
        print('WARNING: REUSE_CACHES_FROM names a run without caches for this arm:', _d)
print('Caches of other runs read by this run:', [d.parent.parent.name for d in _reuse_dirs if d.exists() and d != CACHE_DIR] or 'none')
_reuse_dirs = [d for d in _reuse_dirs if d.exists() and d != CACHE_DIR]
install_call_cache(CACHE_DIR / 'llm_call_cache.jsonl',
                   reuse=[d / 'llm_call_cache.jsonl' for d in _reuse_dirs if (d / 'llm_call_cache.jsonl').exists()])
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)

# ---- 6. Annotation, used by 2A fidelity, 2B, 2C, 2D, 2E, 3A (code/common/annotation.py) -
# One call labels up to 30 hypotheses for one profile; a label missing from the reply is asked
# for again, then asked alone. Prompts: prompts/annotation.txt.
from annotation import build_annotation_matrix as _build_annotation_matrix, set_reuse_cache_dirs, failed_labels
set_reuse_cache_dirs(_reuse_dirs)
build_annotation_matrix = partial(_build_annotation_matrix, mode='binary', model=ANNOT_MODEL,
                                  n_workers=ANNOTATE_WORKERS, style=ANNOTATE_STYLE)

# ---- 7. Compute device (the pipeline runs on CPU; a GPU only speeds up embedding) -----------
try:
    import torch
    print('Device:', 'CUDA - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else 'Apple MPS for the embedder; the SAE trains on CPU' if torch.backends.mps.is_available() else 'CPU')
except Exception as e:
    print('torch is not importable:', e)
print(f'Setup done. Timeout {LLM_TIMEOUT:.0f}s, {ANNOTATE_WORKERS} parallel annotation calls, reply style {ANNOTATE_STYLE!r}, '
      f'Batch API {"on for 2B to 2E and 3A" if ANNOTATION_BATCH_API else "off"}.')

## 3. Phase 0: load the data
Profiles and labels for TRAIN and TEST, a label-stratified subsample of each, and the discovery / confirmation split of TRAIN that Phases 2B to 2E use.

In [ ]:
from clip_h_data import prepare_split, report_splits, stratified_subsample
from self_refine import discovery_confirmation_split

# train.jsonl / test.jsonl = profile text + label per row. Built from *_profile.jsonl + *_labels.csv
# when missing (the build stops if any profile still contains code-status text).
force = FORCE_RERUN['handoff_jsonl']
train_profiles, y_train, ids_train, train_jsonl = prepare_split('train', TASK_DIR, force=force, limit=None)
test_profiles,  y_test,  ids_test,  test_jsonl  = prepare_split('test',  TASK_DIR, force=force, limit=None)
train_profiles, y_train, ids_train = stratified_subsample(train_profiles, y_train, ids_train, N_PATIENTS_TRAIN, SEED)
test_profiles,  y_test,  ids_test  = stratified_subsample(test_profiles,  y_test,  ids_test,  N_PATIENTS_TEST,  SEED)

report_splits({'Train': y_train, 'Test': y_test}, pos_name=POS_NAME, neg_name=NEG_NAME)
print(f'\nSample profile (first 400 chars):\n{train_profiles[0][:400]}')

# Discovery / confirmation split of TRAIN, made once here and used by every later phase:
#   discovery sample  -> Phase 2B (refine), 2C (exemplars), 2D (dedup)
#   confirmation part -> Phase 2E (gate)
_disc_mask, _conf_mask = discovery_confirmation_split(
    len(train_profiles), frac=DISCOVERY_FRAC, seed=SEED, labels=(np.asarray(y_train) if STRATIFY_SPLIT else None))
DISC_IDX, CONF_IDX = np.where(_disc_mask)[0], np.where(_conf_mask)[0]
assert not set(DISC_IDX) & set(CONF_IDX) and len(DISC_IDX) + len(CONF_IDX) == len(train_profiles), \
    'discovery and confirmation must not overlap and must cover TRAIN'
SR_IDX = DISC_IDX if SR_SAMPLE_N is None else DISC_IDX[:SR_SAMPLE_N]
sr_profiles = [train_profiles[i] for i in SR_IDX]   # profiles only: Phases 2B to 2D never read the outcome of the discovery sample
print(f'\nTRAIN split: discovery {len(DISC_IDX)} (sample of {len(SR_IDX)} annotated in 2B/2C/2D), confirmation {len(CONF_IDX)}')


def annotate_discovery_sample(hyps, label):
    """0/1 matrix (discovery-sample profiles x hyps). One cache for Phases 2B, 2C and 2D."""
    return np.asarray(build_annotation_matrix(
        texts=sr_profiles, concepts=list(hyps),
        cache_path=CACHE_DIR / 'selfrefine_annotation_cache.json',
        batch_api=ANNOTATION_BATCH_API, label=label))


write_json(RESULTS_DIR / 'run_manifest.json', {
    'task': TASK_NAME,
    'run_name': RUN_NAME,
    'train_jsonl': str(train_jsonl),
    'test_jsonl': str(test_jsonl),
    'n_train': len(train_profiles),
    'n_test': len(test_profiles),
    'train_label_counts': {str(k): int(v) for k, v in Counter(map(int, y_train)).items()},
    'test_label_counts': {str(k): int(v) for k, v in Counter(map(int, y_test)).items()},
    'train_pos_rate': float(y_train.mean()),
    'test_pos_rate': float(y_test.mean()),
    'model_profile': MODEL_PROFILE,
    'interp_model': INTERP_MODEL,
    'annot_model': ANNOT_MODEL,
    'n_hypothesis_pool': N_HYPOTHESIS_POOL,
    'conversational_reviewers': N_REVIEWERS,
})
print('Wrote', RESULTS_DIR / 'run_manifest.json')

## 4. Phase 1: embeddings, SAE, neuron selection
No API call. Embeddings and the SAE are loaded when their files exist. Selection picks `N_HYPOTHESIS_POOL` neurons related to the outcome and not to each other.

In [ ]:
from embed_profiles import get_local_embeddings, check_embeddings
from clip_h_sae import train_sae, load_sae_pickle, sae_activations, select_neurons, split_weak_neurons

emb_path = PHASE1_DIR / 'embeddings.npz'
sae_path = PHASE1_DIR / 'sae.pkl'

# 1. Embeddings: one vector per profile, from the local embedding model (no API).
if emb_path.exists() and not FORCE_RERUN['embeddings']:
    data = np.load(emb_path)
    train_emb, test_emb = data['train_emb'], data['test_emb']
    print('Loaded embeddings:', emb_path)
else:
    text_to_emb = get_local_embeddings(train_profiles + test_profiles, model=EMB_MODEL, batch_size=32,
                                       cache_name=f'{TASK_NAME}_profiles')
    train_emb = np.vstack([text_to_emb[t] for t in train_profiles])
    test_emb = np.vstack([text_to_emb[t] for t in test_profiles])
    np.savez_compressed(emb_path, train_emb=train_emb, test_emb=test_emb)
    print('Saved embeddings:', emb_path)
check_embeddings(train_emb, len(train_profiles), 'train embeddings')   # stops on a file made for other profiles, or holding NaN
check_embeddings(test_emb, len(test_profiles), 'test embeddings')

# 2. SAE: trained on the TRAIN embeddings; each neuron is a candidate concept.
if sae_path.exists() and not FORCE_RERUN['sae']:
    sae = load_sae_pickle(sae_path)   # works for a file trained on a GPU: it is mapped to the CPU
    print('Loaded SAE:', sae_path)
else:
    sae = train_sae(train_emb, M=SAE_M, K=SAE_K, checkpoint_dir=str(PHASE1_DIR / 'sae_checkpoint'),
                    overwrite_checkpoint=FORCE_RERUN['sae'], n_epochs=SAE_EPOCHS, patience=SAE_PATIENCE, seed=SEED)
    with open(sae_path, 'wb') as f:
        pickle.dump(sae, f)
    print('Saved SAE:', sae_path)

# 3. Neuron selection (no LLM, always recomputed, takes seconds). It uses the labels of ALL
#    training profiles, including those of the confirmation split.
activations, neuron_source_sae_info = sae_activations(sae, train_emb)
selected_neurons, selection_scores = select_neurons(
    activations=activations, target=np.array(y_train), n_select=min(N_HYPOTHESIS_POOL, SAE_M),
    method=SELECTION_METHOD, classification=True, mmr_lambda=MMR_LAMBDA, verbose=True)

# 4. Strong neurons are interpreted one by one (Phase 2A); weak ones are grouped (Phase 2A+).
strong_neurons, strong_scores, WEAK_NEURONS = split_weak_neurons(
    activations, selected_neurons, selection_scores, min_profiles=SPARSE_MAX_PROFILES)
PT.show(None, FUNNEL, 'Phase 1', None, len(selected_neurons),
        f'{len(selected_neurons)} neurons selected: {len(strong_neurons)} strong (fire on >= {SPARSE_MAX_PROFILES} '
        f'train profiles), {len(WEAK_NEURONS)} weak.')

## 5. Phase 2A: interpret the strong neurons
For each strong neuron the LLM writes 3 candidate hypotheses from 30 example profiles; each candidate is scored by how well it reproduces the firing of its neuron; the best one is kept.

In [ ]:
from interpret_neurons import interpret_neurons, score_interpretations, keep_best_interpretations

pool_path = P2A_DIR / 'hypotheses_pool.csv'

if pool_path.exists() and not FORCE_RERUN['hypotheses']:
    pool_df = pd.read_csv(pool_path)
    print('Loaded candidate hypothesis pool:', pool_path)
else:
    # 1. N_CANDIDATES candidate interpretations per strong neuron (INTERP_MODEL, temperature 0; the calls differ in their zero-activation examples).
    #    Prompt: prompts/interpret-neuron-binary.txt, shown 30 profiles per call.
    interpretations = interpret_neurons(
        train_profiles, activations, strong_neurons,
        model=INTERP_MODEL,
        prompt_name='interpret-neuron-binary',
        task_specific_instructions=TASK_INSTRUCTION,
        n_candidates=N_CANDIDATES,
        n_examples=30,
        max_words_per_example=256,
        temperature=0.0,
        max_tokens=_PROFILE['interp_max_tokens'],
        timeout=LLM_TIMEOUT,
        n_workers=_PROFILE['interp_workers'],
        cache_path=CACHE_DIR / 'interpretation_cache.json',   # each finished call is saved at once: an interrupted run resumes
    )
    # 2. Fidelity: each candidate is annotated on 30 profiles (ANNOT_MODEL) and compared with the neuron's firing.
    fidelity_metrics = score_interpretations(
        train_profiles, activations, interpretations,
        annotator_model=ANNOT_MODEL,
        n_examples=30,
        max_words_per_example=256,
        n_workers=ANNOTATE_WORKERS,
        cache_path=CACHE_DIR / 'fidelity_annotation_cache.json',   # keyed on model + hypothesis + whole text
        annotation_style=ANNOTATE_STYLE,
    )
    # 3. One row per neuron: the candidate with the best fidelity F1.
    pool_df = keep_best_interpretations(
        strong_neurons, strong_scores, interpretations, fidelity_metrics, neuron_source_sae_info,
        selection_method=SELECTION_METHOD, scoring_metric='f1')
    pool_df.to_csv(pool_path, index=False)
    print('Saved candidate hypothesis pool:', pool_path)

_t = PT.table_2a(pool_df, activations, ORIGIN)
PT.show(_t, FUNNEL, 'Phase 2A', len(strong_neurons), len(_t),
        f'one hypothesis per strong neuron: the best of {N_CANDIDATES} candidate(s) by fidelity F1.')

## 6. Phase 2A+: group the weak neurons
A weak neuron fires on too few profiles to name a hypothesis alone. Each gets a short name, and the LLM groups clinically related names into one OR-hypothesis per group.

In [ ]:
from weak_groups import interpret_weak_groups

# Grouped rows are stored in hypotheses_pool.csv (neuron_idx -1, -2, ...) and described in pooled_group.json.
_cached_groups = read_json(P2A_DIR / 'pooled_group.json', {})
_groups_cached = (bool(_cached_groups)
                  and bool((pool_df['neuron_idx'].astype(int) < 0).any())
                  and not FORCE_RERUN['hypotheses'])
if _groups_cached:
    POOLED_GROUPS = _cached_groups
    print('Loaded grouped hypotheses:', P2A_DIR / 'pooled_group.json')
elif not WEAK_NEURONS:
    POOLED_GROUPS = {}
    print('No weak neurons, nothing to group.')
else:
    # One call per weak neuron names it; one call sorts the names into clinically related groups
    # and states each group as one OR-hypothesis. No outcome labels are used; a neuron related
    # to no other is left out. Prompts: prompts/weak_groups.txt, run on GROUP_MODEL.
    _group_rows, POOLED_GROUPS = interpret_weak_groups(
        train_profiles, activations, WEAK_NEURONS,
        model=GROUP_MODEL,
        max_tokens=_PROFILE['group_max_tokens'],
        timeout=LLM_TIMEOUT,
        top_per_neuron=5,            # top profiles shown per weak neuron
        n_negatives=15,              # profiles on which the neuron does not fire
        max_chars_per_example=256,   # each profile is cut to this many characters
        negatives_seed=42,
    )
    if _group_rows:
        _rows = []
        for _g in _group_rows:
            row = {c: np.nan for c in pool_df.columns}
            row['neuron_idx'] = _g['neuron_idx']
            if 'source_sae' in pool_df.columns:
                row['source_sae'] = pool_df['source_sae'].iloc[0]
            row['interpretation'] = _g['interpretation']
            _rows.append(row)
        pool_df = pool_df[pool_df['neuron_idx'].astype(int) >= 0].copy()   # replace grouped rows of an earlier run
        pool_df = pd.concat([pool_df, pd.DataFrame(_rows)], ignore_index=True)
        pool_df.to_csv(pool_path, index=False)
        write_json(P2A_DIR / 'pooled_group.json', POOLED_GROUPS)
        print(f'Added {len(_rows)} grouped hypotheses to the pool.')
    else:
        print('WARNING: the model formed no group of 2 or more weak neurons; the pool is unchanged.')

_t = PT.table_2a_plus(pool_df, POOLED_GROUPS, ORIGIN)
PT.show(_t, FUNNEL, 'Phase 2A+', len(WEAK_NEURONS), len(_t),
        f'weak neurons in, grouped hypotheses out; the pool now holds {len(pool_df)} hypotheses.')

## 7. Fidelity gate
No LLM call. Drops hypotheses below `MIN_FIDELITY` and exact duplicates. Grouped hypotheses have no fidelity score and pass.

In [ ]:
from fidelity_gate import curate

candidates_path = P2A_DIR / 'hypotheses_curated_candidates.csv'
curation_path = P2A_DIR / 'curation_summary.json'

if candidates_path.exists() and curation_path.exists() and not FORCE_RERUN['hypotheses']:
    candidates_df = pd.read_csv(candidates_path)
    curation_summary = read_json(curation_path, {})
    print('Loaded curated candidates:', candidates_path)
else:
    # No LLM: drop hypotheses below MIN_FIDELITY, drop exact text duplicates, rank.
    candidates_df, curation_summary = curate(pool_df, MIN_FIDELITY)
    candidates_df.to_csv(candidates_path, index=False)
    write_json(curation_path, curation_summary)
    print('Saved curated candidates:', candidates_path)

_t = PT.table_gate(pool_df, curate(pool_df, MIN_FIDELITY)[0], MIN_FIDELITY, ORIGIN)   # recomputed for the table only
PT.show(_t, FUNNEL, 'Fidelity gate', len(pool_df), curation_summary['curated_candidate_count'],
        f"{curation_summary['dropped_low_fidelity']} below fidelity {MIN_FIDELITY}, "
        f"{curation_summary['exact_merged']} with the same text as another.")

## 8. Phase 2B: self-refine
On the discovery sample only. The critic never sees the outcome. Skipped when `CONTROL_LOOP` is False.

In [ ]:
import time
import self_refine as SR

SELFREFINE_HYP_PATH   = P2B_DIR / 'self_refine_hypotheses.csv'
SELFREFINE_TRACE_PATH = P2B_DIR / 'self_refine_trace.json'

if not CONTROL_LOOP:
    print('CONTROL_LOOP=False: Phase 2B skipped,', len(candidates_df), 'candidates forwarded unchanged.')
elif SELFREFINE_HYP_PATH.exists() and not FORCE_RERUN['self_refine']:
    candidates_df = pd.read_csv(SELFREFINE_HYP_PATH)
    print('Loaded self-refined hypotheses:', SELFREFINE_HYP_PATH, f'({len(candidates_df)})')
else:
    _t0 = time.time()
    _hyps0 = candidates_df['interpretation'].astype(str).tolist()
    print(f'Phase 2B: {len(_hyps0)} candidates on {len(sr_profiles)} discovery profiles, up to {REFINE_ITERS} rounds, '
          f'refiner and critic = {REFINE_MODEL}')

    def _annotate(hs):
        M = annotate_discovery_sample(hs, 'Phase 2B annotation')
        return {h: M[:, k].astype(float) for k, h in enumerate(hs)}

    # Each round: annotate -> flag hypotheses that are too rare -> critic (accept / sharpen / split /
    # reground / drop, never sees the outcome) -> refiner rewrites. Prompts: prompts/self_refine.txt.
    # An error here stops the cell and nothing is saved, so the next run tries again.
    _refine = SR.make_refine_fn_agentic(REFINE_MODEL, timeout=600, seed=SEED)
    _critic = SR.make_critic_fn(REFINE_MODEL, task=TASK_DESCRIPTION, pos_label=POS_LABEL, timeout=600, seed=SEED)
    refined, sr_trace = SR.agentic_refine(_hyps0, len(sr_profiles), _annotate, _refine,
                                          critic_fn=_critic, iters=REFINE_ITERS, min_prev=SR_MIN_PREV)
    for tr in sr_trace:
        print(f"    round {tr['iter'] + 1}: {tr['n']} hypotheses, {tr['n_niche']} too rare, "
              f"{tr['n_dropped']} dropped by the critic, {tr['n_critic_edits']} edited"
              + (' | the critic gave no usable reply' if tr.get('critic_failed') else '')
              + (' | the refiner gave no usable reply' if tr.get('refine_failed') else ''))
    if refined:
        candidates_df = pd.DataFrame({'interpretation': refined})
    candidates_df = candidates_df.reset_index(drop=True)
    candidates_df.to_csv(SELFREFINE_HYP_PATH, index=False)
    write_json(SELFREFINE_TRACE_PATH, {'trace': sr_trace, 'n_in': len(_hyps0), 'n_out': len(candidates_df),
                                       'discovery_n': int(len(DISC_IDX)), 'confirmation_n': int(len(CONF_IDX))})
    print(f'Phase 2B done in {time.time() - _t0:.0f}s: {len(_hyps0)} -> {len(candidates_df)} hypotheses')

candidates_df = candidates_df.reset_index(drop=True)   # the review numbers the candidates 0..N-1

if CONTROL_LOOP:   # what each hypothesis became, read from the saved trace
    _trace = read_json(SELFREFINE_TRACE_PATH, {}).get('trace', [])
    _t = PT.table_2b(_trace, candidates_df['interpretation'].astype(str).tolist(), ORIGIN)
    PT.show(_t, FUNNEL, 'Phase 2B', (_trace[0]['n'] if _trace else None), len(candidates_df),
            ', '.join(f'{v} {k}' for k, v in _t['status'].value_counts().items()) + '.' if len(_t) else 'no trace saved.')

## 9. Phase 2C: write the review prompt (manual step follows)
Writes `CONVO_01_critique_prompt.md`. Paste the whole file into a chat model, which acts as `N_REVIEWERS` clinical reviewers, and save its JSON reply as `CONVO_02_critique_response.json` in the same folder.

In [ ]:
import conversational_review
from conversational_review import build_concept_exemplars, build_review_prompt

CONVO_PROMPT_PATH = P2C_DIR / 'CONVO_01_critique_prompt.md'
CONVO_RESPONSE_PATH = P2C_DIR / 'CONVO_02_critique_response.json'

# For each candidate, the raw values of up to 3 discovery patients for whom it is true (printed by
# the RAW_FACTS function), so the reviewer can see whether the hypothesis matches the data. Uses
# the Phase 2B labels (same profiles, same cache), so normally no new call.
_firing = annotate_discovery_sample(candidates_df['interpretation'].astype(str).tolist(), 'Phase 2C annotation')
train_features = pd.read_csv(TASK_DIR / 'train_features.csv')
_sr_features = train_features.iloc[[int(ids_train[i]) for i in SR_IDX]].reset_index(drop=True)   # same rows as sr_profiles
concept_exemplars = build_concept_exemplars(
    candidates_df, _firing, _sr_features, k=3, withhold=WITHHOLD_RAW_FIELDS,
    fact_fn=getattr(conversational_review, RAW_FACTS))

# Prompt = prompts/review.txt + the numbered candidates. Always rewritten.
review_prompt = build_review_prompt(candidates_df, load_prompt('review'), n_reviewers=N_REVIEWERS, exemplars=concept_exemplars)
CONVO_PROMPT_PATH.write_text(review_prompt, encoding='utf-8')
print('Wrote:', CONVO_PROMPT_PATH)
print('Candidates:', len(candidates_df), '| with raw-value exemplars:', sum(1 for v in concept_exemplars.values() if v),
      '| withheld fields:', sorted(WITHHOLD_RAW_FIELDS) or 'none')
print('NEXT (manual): paste the whole file into a chat model and save its JSON reply as', CONVO_RESPONSE_PATH)

## 10. Phase 2C+: read the review reply
Stops with instructions until the reply file exists.

In [ ]:
from conversational_review import ingest_review_response

if not CONVO_RESPONSE_PATH.exists():
    raise FileNotFoundError(
        'Manual step not done yet. Paste ' + str(CONVO_PROMPT_PATH) + ' into a chat model and save its '
        'JSON-only reply as ' + str(CONVO_RESPONSE_PATH) + ', then run this cell again.')

# Checks the reply (strict JSON, score ranges, vote totals, every candidate covered exactly once, no
# outcome words), then keeps a hypothesis with a majority of valid votes, clinical validity >=
# MIN_CLINICAL_VALIDITY and leakage risk < LEAKAGE_DROP.
final_df, conversational_merged_df, conversational_summary = ingest_review_response(
    CONVO_RESPONSE_PATH,
    candidates_df,
    P2C_DIR,
    n_reviewers=N_REVIEWERS,
    max_hypotheses=N_HYPOTHESIS_POOL,   # no quota: at most the pool size
    min_clinical_validity=MIN_CLINICAL_VALIDITY,
    leakage_drop=LEAKAGE_DROP,
)
hypotheses = final_df['interpretation'].astype(str).tolist()
_t = PT.table_2c(conversational_merged_df, candidates_df['interpretation'].astype(str).tolist(), ORIGIN,
                 n_reviewers=N_REVIEWERS, min_clinical_validity=MIN_CLINICAL_VALIDITY, leakage_drop=LEAKAGE_DROP)
_s = conversational_summary
PT.show(_t, FUNNEL, 'Phase 2C+', len(candidates_df), len(final_df),
        f"the reviewers turned {_s['candidate_count']} candidates into {_s['merged_concept_count']}; dropped: "
        f"{_s['dropped_minority_vote']} minority vote, {_s['dropped_leakage']} leakage, {_s['dropped_low_validity']} low validity.")

## 11. Phase 2D: merge duplicates
Uses the discovery sample. The kept hypothesis of a merged group is the one with the higher clinical validity.

In [ ]:
import dedup_lib as DL

_dd_labels = final_df['interpretation'].astype(str).tolist()
_dd_dirs = (final_df['expected_direction'].astype(str).tolist()
            if 'expected_direction' in final_df.columns else ['unclear'] * len(_dd_labels))

if len(_dd_labels) > 1:
    # Two hypotheses are duplicates when they are true for nearly the same discovery patients (phi)
    # and the review gave them the same direction. phi >= DEDUP_AUTO_THR merges at once; between
    # DEDUP_CORR_THR and DEDUP_AUTO_THR one DEDUP_MODEL call confirms or refuses (prompts/dedup.txt).
    _M = annotate_discovery_sample(_dd_labels, 'Phase 2D annotation')
    _adj = DL.make_dedup_adjudicator(DEDUP_MODEL, task=TASK_DESCRIPTION, pos_label=POS_LABEL, seed=SEED)
    DEDUP_RESULT = DL.predictive_dedup([_M[:, k].astype(float) for k in range(len(_dd_labels))], _dd_labels, _dd_dirs,
                                       meta=final_df.to_dict('records'),
                                       corr_thr=DEDUP_CORR_THR, auto_thr=DEDUP_AUTO_THR, adjudicate_fn=_adj)
    _kept = DEDUP_RESULT['kept']
    DL.log_stage_table(str(P2D_DIR), 'dedup_pairs',
                       DEDUP_RESULT['pairs'] or [{'note': 'no pair reached DEDUP_CORR_THR'}])
    DL.log_stage_table(str(P2D_DIR), 'dedup_clusters',
                       [{'representative': c['representative'], 'members': ' | '.join(c['members'])}
                        for c in DEDUP_RESULT['clusters']] or [{'representative': '', 'members': ''}])
    final_df = (final_df[final_df['interpretation'].astype(str).isin(_kept)]
                .drop_duplicates('interpretation').reset_index(drop=True))
    for c in DEDUP_RESULT['clusters']:
        _others = [m for m in c['members'] if m != c['representative']]
        print('    kept "' + c['representative'] + '"  <=  ' + ' + '.join('"' + m + '"' for m in _others))
    _t = PT.table_2d(_dd_labels, _M, DEDUP_RESULT, ORIGIN)
    PT.show(_t, FUNNEL, 'Phase 2D', len(_dd_labels), len(_kept),
            f'{len(DEDUP_RESULT["clusters"])} merged group(s); a pair is looked at from phi {DEDUP_CORR_THR}.')
else:
    print('Dedup: one hypothesis or none, nothing to merge.')

hypotheses = final_df['interpretation'].astype(str).tolist()

## 12. Phase 2E: confirmation gate
A hypothesis is certified only if, on the confirmation split, it is FDR-significant, stays so with the same sign in `GATE_STABILITY` of the bootstrap resamples, and is true for at least `ESTIMABILITY_FLOOR` profiles.

In [ ]:
from confirmation_gate import confirmation_gate

gate_path = P2E_DIR / 'confirmation_gate_tests.csv'
gate_summary_path = P2E_DIR / 'confirmation_gate_summary.json'

if not RUN_TRAIN_GATE:
    print('RUN_TRAIN_GATE=False: gate skipped, all', len(hypotheses), 'hypotheses forwarded.')
    train_gate_summary = {'enabled': False, 'n_kept': len(hypotheses)}
else:
    # The confirmation split is the part of TRAIN that Phases 2B-2D never annotated or refined on.
    _conf_idx = CONF_IDX if GATE_SAMPLE_N is None else CONF_IDX[:GATE_SAMPLE_N]
    gate_texts = [train_profiles[i] for i in _conf_idx]
    gate_y = np.asarray(y_train, dtype=int)[_conf_idx]
    # The only paid step of this phase: annotate the hypotheses on the confirmation profiles.
    gate_M = build_annotation_matrix(
        texts=gate_texts, concepts=hypotheses,
        cache_path=CACHE_DIR / 'binary_confirmation_annotation_cache.json',
        batch_api=ANNOTATION_BATCH_API, label='Phase 2E annotation')
    # No LLM from here: association test + Benjamini-Hochberg FDR, then bootstrap stability.
    kept, gate_tests, train_gate_summary = confirmation_gate(
        gate_M, gate_y, hypotheses,
        fdr_alpha=GATE_FDR_ALPHA, n_bootstrap=GATE_BOOTSTRAP, stability_threshold=GATE_STABILITY,
        estimability_floor=ESTIMABILITY_FLOOR, min_survivors=MIN_SURVIVORS, top_k=GATE_TOP_K, seed=SEED,
        verbose=False)   # the table below shows every hypothesis
    gate_tests.to_csv(gate_path, index=False)
    final_df = final_df[final_df['interpretation'].astype(str).isin(kept)].reset_index(drop=True)
    hypotheses = final_df['interpretation'].astype(str).tolist()

gate_summary_path.write_text(json.dumps(train_gate_summary, indent=2), encoding='utf-8')
if RUN_TRAIN_GATE:
    _t = PT.table_2e(gate_tests, train_gate_summary, kept, ORIGIN, fdr_alpha=GATE_FDR_ALPHA,
                     stability_threshold=GATE_STABILITY, estimability_floor=ESTIMABILITY_FLOOR, outcome=POS_NAME)
    PT.show(_t, FUNNEL, 'Phase 2E', len(gate_tests), len(kept),
            f'tested on {len(gate_texts)} confirmation patients. Certified = true for at least {ESTIMABILITY_FLOOR}, '
            f'q at most {GATE_FDR_ALPHA}, stability at least {GATE_STABILITY}.')

## 13. Phase 3A: annotate and predict
The reported AUROC, AUPRC and F1 come from a logistic regression fit on TRAIN annotations and scored on TEST.

In [ ]:
from evaluation import score_hypotheses, compute_hypothesis_association_tests, fit_and_score_heldout, bootstrap_intervals

annotation_path = P3_DIR / 'annotation_matrix.npz'
evaluation_path = P3_DIR / 'hypothesis_evaluation.csv'
metrics_path = P3_DIR / 'annotation_metrics.json'
association_path = P3_DIR / 'hypothesis_association_tests.csv'
association_summary_path = P3_DIR / 'hypothesis_association_summary.json'
heldout_path = P3_DIR / 'heldout_metrics.json'
model_metrics_path = P3_DIR / 'model_metrics.csv'
_force = FORCE_RERUN['annotation']

# 1. TEST annotations + per-hypothesis statistics on the test set (coefficients and p-values;
#    these are fit on the test set itself, so they describe hypotheses, not prediction quality).
if annotation_path.exists() and evaluation_path.exists() and metrics_path.exists() and not _force:
    annotation_matrix = np.load(annotation_path, allow_pickle=True)['annotation_matrix']
    annotation_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    evaluation_df = pd.read_csv(evaluation_path)
    print('Loaded test annotations and per-hypothesis statistics.')
else:
    annotation_matrix = build_annotation_matrix(
        texts=test_profiles, concepts=hypotheses,
        cache_path=CACHE_DIR / 'binary_annotation_cache.json',
        batch_api=ANNOTATION_BATCH_API, label='Phase 3A test annotation')
    annotation_metrics, evaluation_df = score_hypotheses(
        {hypothesis: annotation_matrix[:, column] for column, hypothesis in enumerate(hypotheses)},
        np.asarray(y_test, dtype=int), classification=True, corrected_pval_threshold=0.1)
    evaluation_df.to_csv(evaluation_path, index=False)
    metrics_path.write_text(json.dumps(json_safe(annotation_metrics), indent=2), encoding='utf-8')
    np.savez_compressed(annotation_path, annotation_matrix=annotation_matrix,
                        hypotheses=np.array(hypotheses, dtype=object), labels=np.asarray(y_test, dtype=int))

# 2. TRAIN annotations of the final hypotheses, in their own label cache. The confirmation patients are
#    annotated here again: Phase 2E asked about them with the larger set of hypotheses it tested.
n_eval_train = len(train_profiles) if GATE_TRAIN_N is None else min(GATE_TRAIN_N, len(train_profiles))
X_train_eval = build_annotation_matrix(
    texts=train_profiles[:n_eval_train], concepts=hypotheses,
    cache_path=CACHE_DIR / 'binary_train_annotation_cache.json',
    batch_api=ANNOTATION_BATCH_API, label='Phase 3A train annotation').astype(float)

# 3. The headline numbers: logistic regression fit on TRAIN annotations, scored on TEST.
heldout_metrics, test_score, model_coef = fit_and_score_heldout(
    X_train_eval, np.asarray(y_train, dtype=int)[:n_eval_train], annotation_matrix, y_test, seed=SEED)
heldout_metrics['ci95'] = bootstrap_intervals(y_test, test_score, seed=SEED)   # 10,000 resamples of the test set, model not refit
heldout_path.write_text(json.dumps(heldout_metrics, indent=2), encoding='utf-8')

# 4. 2x2 association test of every hypothesis with the outcome on the test set (no LLM).
if association_path.exists() and association_summary_path.exists() and not _force:
    hypothesis_association_summary = json.loads(association_summary_path.read_text(encoding='utf-8'))
    association_df = pd.read_csv(association_path)
    print('Loaded association tests.')
else:
    _npz = np.load(annotation_path, allow_pickle=True)
    hypothesis_association_summary, association_df = compute_hypothesis_association_tests(
        annotation_matrix=_npz['annotation_matrix'], labels=_npz['labels'],
        hypotheses=[str(h) for h in _npz['hypotheses'].tolist()], fdr_alpha=0.05)
    association_df.to_csv(association_path, index=False)
    association_summary_path.write_text(json.dumps(json_safe(hypothesis_association_summary), indent=2), encoding='utf-8')
    print('Saved association tests:', association_path)

# 5. One-row metrics table of this run.
model_metrics_df = pd.DataFrame([{
    'run_name': RUN_NAME, 'task': TASK_NAME, 'model': 'clip_h',
    'n_train': int(len(y_train)), 'n_test': int(len(y_test)),
    'train_positives': int(np.asarray(y_train, dtype=int).sum()),
    'test_positives': int(np.asarray(y_test, dtype=int).sum()),
    **{k: heldout_metrics[k] for k in ('auroc', 'auprc', 'f1', 'precision', 'recall')},
    'source_metrics_path': str(heldout_path),
}])
model_metrics_df.to_csv(model_metrics_path, index=False)

# 6. What to read: the held-out result, then the final hypotheses.
print(f'Held-out result: logistic regression fit on {n_eval_train} train patients, scored on {len(y_test)} test patients, '
      f'{len(hypotheses)} hypotheses.')
PT.show(PT.metrics_table(heldout_metrics))
_t = PT.table_3a(hypotheses, annotation_matrix, association_df, model_coef, ORIGIN, outcome=POS_NAME)
PT.show(_t, FUNNEL, 'Phase 3A', len(hypotheses), len(hypotheses),
        'the final hypotheses on the test set, largest model weight first.')
print(f"For reference only, a model fit on the test set itself (describes the hypotheses, is not the result): "
      f"AUROC {annotation_metrics['auroc']:.3f}. Full per-hypothesis files: {evaluation_path.name}, {association_path.name}")

## 14. Phase 3B: summary
Checks that every expected file exists, writes the summary and the file index, prints the counts of every phase, the held-out result and the cost.

In [ ]:
expected_outputs = {
    'representation_and_sae': [PHASE1_DIR / 'embeddings.npz', PHASE1_DIR / 'sae.pkl'],
    'candidate_generation': [P2A_DIR / 'hypotheses_pool.csv', P2A_DIR / 'hypotheses_curated_candidates.csv',
                             P2A_DIR / 'curation_summary.json'],
    'conversational_review': [P2C_DIR / 'CONVO_01_critique_prompt.md', P2C_DIR / 'CONVO_02_critique_response.json',
                              P2C_DIR / 'hypotheses_conversational_merged.csv', P2C_DIR / 'hypotheses.csv',
                              P2C_DIR / 'conversational_curation_summary.json'],
    'profile_annotation_evaluation': [annotation_path, evaluation_path, metrics_path, association_path,
                                      association_summary_path],
    'model_metrics': [model_metrics_path, heldout_path],
}
artifact_index = {
    group: [{'path': str(path), 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0}
            for path in paths]
    for group, paths in expected_outputs.items()
}
missing = [item['path'] for items in artifact_index.values() for item in items if not item['exists']]
if missing:
    raise FileNotFoundError('Missing expected files:\n' + '\n'.join(missing))

write_json(RESULTS_DIR / 'task_summary.json', {
    'task': TASK_NAME,
    'status': 'ok',
    'output_dir': str(RESULTS_DIR),
    'n_hypotheses': len(hypotheses),
    'review_source': 'conversational',
    'annotation_model': ANNOT_MODEL,
    'conversational_curation': conversational_summary,
    'annotation_metrics': json_safe(annotation_metrics),
    'heldout_metrics': json_safe(heldout_metrics),
    'model_metrics_path': str(model_metrics_path),
    'hypothesis_association_summary': json_safe(hypothesis_association_summary),
    'annotation_failures': failed_labels(CACHE_DIR),   # labels that could not be obtained and count as 0, per step
    'artifact_index': artifact_index,
})
write_json(RESULTS_DIR / 'results_index.json', artifact_index)

print('=' * 70)
print(f'RUN SUMMARY  |  run={RUN_NAME}  task={TASK_NAME}  arm={MODEL_PROFILE}')
print('=' * 70)
print('\n1. Hypotheses left after each phase')
PT.show(PT.funnel_table(FUNNEL))
print(f'\n2. Held-out result: {len(hypotheses)} hypotheses, fit on {n_eval_train} train patients, scored on {len(y_test)} test patients')
PT.show(PT.metrics_table(heldout_metrics))
_arms = {arm: read_json(RUN_DIR / f'{TASK_NAME}_{arm}' / 'phase3_prediction' / 'heldout_metrics.json', None)
         for arm in _MODEL_PROFILES}
_arms = {arm: m for arm, m in _arms.items() if m}
if len(_arms) > 1:
    print('\n3. The arms of this run side by side: value (95% interval)')
    PT.show(PT.arms_table(_arms))
print('\nThe final hypotheses are in the Phase 3A table above.')
_failed = {step: n['failed'] for step, n in failed_labels(CACHE_DIR).items() if n['failed']}
print('\nLabels that could not be obtained and count as 0:', _failed or 'none')
print('\nCost of this arm so far:')
usage_report(CACHE_DIR / 'usage_log.jsonl')
print(f'\nResults are in: {RESULTS_DIR}')